<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/ai/04_generative_ai/llms/experiment_bert_tokenization_embeddings_similarity_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# LEVEL 1: TOKENIZATION
# ============================================

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "I love AI"

tokens = tokenizer.tokenize(text)
ids = tokenizer.encode(text)

print("Tokens:", tokens)
print("Token IDs:", ids)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokens: ['i', 'love', 'ai']
Token IDs: [101, 1045, 2293, 9932, 102]


In [ ]:
# ============================================
# LEVEL 2: EMBEDDINGS
# ============================================

from transformers import AutoModel
import torch

model = AutoModel.from_pretrained("bert-base-uncased")

inputs = tokenizer("I love AI", return_tensors="pt")

outputs = model(**inputs)

embeddings = outputs.last_hidden_state

print("Embedding shape:", embeddings.shape)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding shape: torch.Size([1, 5, 768])


In [ ]:
sentence_embedding = embeddings.mean(dim=1)

print("Sentence vector:", sentence_embedding.shape)

Sentence vector: torch.Size([1, 768])


In [ ]:
import torch.nn.functional as F

s1 = tokenizer("I love AI", return_tensors="pt")
s2 = tokenizer("I like machine learning", return_tensors="pt")

e1 = model(**s1).last_hidden_state.mean(dim=1)
e2 = model(**s2).last_hidden_state.mean(dim=1)

similarity = F.cosine_similarity(e1, e2)

print("Similarity:", similarity.item())

Similarity: 0.6657829284667969


In [ ]:
sentences = [
    "I love AI",
    "Machine learning is amazing",
    "I hate bugs",
    "Deep learning is powerful"
]

query = "AI is great"

query_emb = model(**tokenizer(query, return_tensors="pt")).last_hidden_state.mean(dim=1)

scores = []

for s in sentences:
    emb = model(**tokenizer(s, return_tensors="pt")).last_hidden_state.mean(dim=1)
    score = F.cosine_similarity(query_emb, emb).item()
    scores.append(score)

best_idx = scores.index(max(scores))

print("Best match:", sentences[best_idx])

Best match: Machine learning is amazing
